# Wiki-RAG Demo

End-to-end demo of the Wiki-RAG pipeline: a Retrieval-Augmented Generation system that answers questions over a curated set of Machine Learning / AI Wikipedia articles, running entirely locally via Ollama.

This notebook is intentionally thin — all real logic lives in `src/`. The notebook just demonstrates the pipeline end-to-end.

**Prerequisites:** Ollama installed and running, with `llama3.2:3b` and `nomic-embed-text` pulled (`ollama pull llama3.2:3b && ollama pull nomic-embed-text`).

In [ ]:
import sys
sys.path.append('..')

from src.config import Config
from src.pipeline import RAGPipeline

## 1. Build the index

This fetches ~40 ML/AI Wikipedia articles (cached locally after the first run), chunks them, embeds each chunk with `nomic-embed-text`, and stores the vectors in a local ChromaDB collection.

In [ ]:
config = Config()
pipeline = RAGPipeline(config)
pipeline.build_index()

## 2. Ask questions

In [ ]:
result = pipeline.ask("What is the difference between supervised and unsupervised learning?")
print(result.answer)
print()
for src in result.sources:
    print(f"- {src.source_title} (distance={src.distance:.3f})")

In [ ]:
questions = [
    "What is overfitting and how can it be prevented?",
    "Explain the attention mechanism in transformers.",
    "What is the bias-variance tradeoff?",
]

for q in questions:
    result = pipeline.ask(q)
    print(f"Q: {q}\nA: {result.answer}\n")

## 3. Evaluate retrieval quality

A small hand-labeled eval set checks whether the retriever surfaces context containing the expected keyword for each question — a cheap but honest precision@k signal.

In [ ]:
from src.evaluate import evaluate_retrieval

results = evaluate_retrieval(config)
print(f"Retrieval precision@k: {results['precision_at_k']:.0%}")
results['details']